# Notebook 05: Backtest Results

Full backtest analysis:
- **Equity curve**: cumulative portfolio value over time
- **Monthly returns heatmap**: returns by year × month
- **Drawdown**: when and how deep losses occurred
- **Risk metrics**: Sharpe ratio, Sortino ratio, max drawdown

In [ ]:
import sys
sys.path.append('..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

from src.factors.momentum import MomentumFactor
from src.factors.mean_reversion import MeanReversionFactor
from src.factors.volatility import VolatilityFactor
from src.factors.adaptive_composite import AdaptiveCompositeFactor, AdaptiveCompositeManager

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline
print('✅ Imports successful')

## 1. Load Demo Data

In [ ]:
raw_dir = Path('../data/raw')
parquet_files = sorted(raw_dir.glob('*.parquet'))
demo_file = [f for f in parquet_files if '10tickers' in f.name] or [parquet_files[0]]
data = pd.read_parquet(demo_file[0])
print(f'Shape: {data.shape}')

## 2. Compute Factor Signals

In [ ]:
factors = [
    MomentumFactor(lookback=126, name='Momentum'),
    MeanReversionFactor(lookback=21, name='MeanReversion'),
    VolatilityFactor(window=63, name='Volatility'),
]
af = AdaptiveCompositeFactor(factors=factors, ic_window=63, decay_halflife=21, min_weight=0.05)
manager = AdaptiveCompositeManager(af, forward_period=21)

all_dates = sorted(data.index.get_level_values('date').unique())
factor_signals = {}

print(f'Computing factors for {len(all_dates)} dates...')
for i, date in enumerate(all_dates):
    date = pd.Timestamp(date)
    data_slice = data[data.index.get_level_values('date') <= date]
    prices_wide = data['close'].unstack('ticker')
    prices_today = prices_wide[prices_wide.index <= date].iloc[-1]
    try:
        composite, _ = manager.compute_factor_with_update(data_slice, prices_today, date)
        factor_signals[date] = composite[composite.index.get_level_values('date') == date].droplevel('date')
    except Exception:
        pass
    if (i + 1) % 200 == 0:
        print(f'  {i+1}/{len(all_dates)}')

print(f'✅ Factor signals computed for {len(factor_signals)} dates')

## 3. Run Backtest

In [ ]:
close_wide = data['close'].unstack('ticker')
tickers = close_wide.columns.tolist()

INITIAL_CAPITAL = 1_000_000
TRANSACTION_COST = 0.001
REBALANCE_FREQ = 21

portfolio_value = [INITIAL_CAPITAL]
daily_returns = []
positions = pd.Series(0.0, index=tickers)
rebalance_dates = []

signal_dates = sorted(factor_signals.keys())
for i, date in enumerate(signal_dates[1:], 1):
    prev_date = signal_dates[i - 1]

    price_change = (close_wide.loc[date] / close_wide.loc[prev_date] - 1).fillna(0)
    daily_pnl = (positions * price_change).sum()
    portfolio_value.append(portfolio_value[-1] * (1 + daily_pnl))
    daily_returns.append(daily_pnl)

    if i % REBALANCE_FREQ == 0:
        signal = factor_signals[date].reindex(tickers).fillna(0)
        n_long = max(1, len(tickers) // 4)
        top = signal.nlargest(n_long).index
        new_positions = pd.Series(0.0, index=tickers)
        new_positions[top] = 1.0 / n_long
        turnover = (new_positions - positions).abs().sum() / 2
        cost = turnover * TRANSACTION_COST
        portfolio_value[-1] *= (1 - cost)
        positions = new_positions
        rebalance_dates.append(date)

pv = pd.Series(portfolio_value, index=[signal_dates[0]] + signal_dates[1:])
ret = pd.Series(daily_returns, index=signal_dates[1:])
print(f'Backtest complete: {len(pv)} days')
print(f'Final portfolio value: ${pv.iloc[-1]:,.0f}')
print(f'Total return: {(pv.iloc[-1]/pv.iloc[0]-1)*100:.1f}%')

## 4. Equity Curve

In [ ]:
benchmark_ret = close_wide.pct_change().fillna(0).mean(axis=1)
benchmark = (1 + benchmark_ret).cumprod() * INITIAL_CAPITAL
benchmark = benchmark.reindex(pv.index, method='ffill')

fig, ax = plt.subplots(figsize=(14, 5))
pv.plot(ax=ax, label='Strategy', linewidth=2)
benchmark.plot(ax=ax, label='Equal-Weight Benchmark', linewidth=1.5, linestyle='--', alpha=0.8)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_title('Equity Curve')
ax.set_ylabel('Portfolio Value ($)')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Monthly Returns Heatmap

In [ ]:
monthly_ret = ret.resample('ME').apply(lambda x: (1 + x).prod() - 1)
monthly_table = monthly_ret.to_frame('ret')
monthly_table['year'] = monthly_table.index.year
monthly_table['month'] = monthly_table.index.month
pivot = monthly_table.pivot(index='year', columns='month', values='ret')
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
pivot.columns = [month_names[m-1] for m in pivot.columns]

fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(pivot * 100, annot=True, fmt='.1f', center=0, cmap='RdYlGn',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Return (%)'})
ax.set_title('Monthly Returns Heatmap (%)')
plt.tight_layout()
plt.show()

## 6. Drawdown

In [ ]:
rolling_max = pv.cummax()
drawdown = (pv - rolling_max) / rolling_max * 100

fig, ax = plt.subplots(figsize=(14, 4))
drawdown.plot(ax=ax, color='red', alpha=0.7)
ax.fill_between(drawdown.index, drawdown, 0, alpha=0.3, color='red')
ax.set_title('Portfolio Drawdown (%)')
ax.set_ylabel('Drawdown (%)')
plt.tight_layout()
plt.show()

## 7. Risk Metrics

In [ ]:
rfr_daily = 0.02 / 252
ann_ret = (pv.iloc[-1] / pv.iloc[0]) ** (252 / len(pv)) - 1
ann_vol = ret.std() * np.sqrt(252)
sharpe = (ann_ret - 0.02) / ann_vol

downside = ret[ret < rfr_daily]
sortino = (ann_ret - 0.02) / (downside.std() * np.sqrt(252) + 1e-9)
max_dd = drawdown.min()

print('='*40)
print('  RISK METRICS')
print('='*40)
print(f'  Annual Return:    {ann_ret*100:.2f}%')
print(f'  Annual Volatility:{ann_vol*100:.2f}%')
print(f'  Sharpe Ratio:     {sharpe:.3f}')
print(f'  Sortino Ratio:    {sortino:.3f}')
print(f'  Max Drawdown:     {max_dd:.2f}%')
print(f'  # Rebalances:     {len(rebalance_dates)}')
print('='*40)